# Suite LAY — Layers' interpreters: the base class, and a layer not built on proxies

Each layer of a stack interprets a transaction with an interpreter of its own, an instance of a class coupled with the
layer, which reads and writes the layer directly, not through the Store interface
([the stacked stores proposal, stacking by mutation transactions](https://github.com/pitaman71/mbse-basis/blob/main/proposals/STACKED-STORES.md#stacking-by-mutation-transactions)). The base class,
`Transactions.Interpreter`, holds what every layer interprets alike: the walk of a transaction's terms (map, upsert,
update, reduce, partition, then), binding symbols, frames, errors annotated on frames
([CAP-309: errors annotated on frames](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-309)), unique clauses checked at commit, and the stream `txn.exec` gives, with its
scopes, remainders, completion markers and journals. A kind of layer's class gives it what only that layer knows: its
reads of its own storage (the candidates for a symbol's schema, the object a reference designates, the entries that
link an object), its writes, and its transaction's savepoints. A store in memory's is `Transactions.InMemory`.

This suite's layer, `Indexed`, holds its data in plain dictionaries, as an index would: each object a row, by an id of
its own, found by its reference's path through a dictionary keyed by the path; each relation's entries a list of rows
linking objects by their ids. It is the test's, not shipped, and interprets with the base class alone, over the schemas
of a `Builders.Store`, so that the same transactions as suites TXN and EXE give the same frames, by their references,
and leave the same data as a store in memory does.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import contextlib
import itertools
import json
from types import SimpleNamespace

from mbse.Schemas.Framework import Builders as B, Changes, Plain, Provenance, References, Transactions
from mbse.Schemas.Framework.Transactions import Frame
from support import raises


class Indexed(Transactions.Interpreter):
    """A layer kept in plain dictionaries, as an index would keep it, interpreted by the base class alone: each object a
    row, `{"id", "schema", "path", "values"}`, its properties' values plain, by an id of its own; `paths`, each object
    that has a reference, by its path's JSON; and each relation's entries, by its name, a list of rows
    `{"links", "values"}`, linking objects by their ids. Its transaction keeps a log of how to undo each write, and one
    of what each wrote, from which it gathers its journal and records it, by what every layer shares (`Changes.gather`,
    `Provenance.record`)."""

    def __init__(self, schemas):
        super().__init__(schemas)
        self.objects, self.paths, self.relations, self.counted = {}, {}, {}, {}
        self.ids = itertools.count()
        self.undo = None  # the transaction open's undo log; None where none is open
        self.inserted, self.missed, self.txn_uid, self.log = [], set(), None, []
        for name in schemas.names():  # each singleton, held from the start
            schema = schemas.registered(name)
            if isinstance(schema, B.S.OfObject.Data) and schema.singleton is not None:
                self._row(name, {"singleton": schema.singleton, "next": {"target": {}}}, logged=False)

    # --- its storage ---

    def _row(self, schema_name, path=None, values=None, logged=True):
        row = {"id": next(self.ids), "schema": schema_name, "path": None, "values": dict(values or {})}
        self.objects[row["id"]] = row
        if logged:
            self.undo.append(lambda: self.objects.pop(row["id"]))
            self.log.append(("create", row))
        if path is not None:
            self._place(row, path, logged)
        return row

    def _place(self, row, path, logged=True):
        old, key = row["path"], json.dumps(path, sort_keys=True)
        row["path"] = path
        self.paths[key] = row["id"]
        if logged:
            self.undo.append(lambda: (self.paths.pop(key), row.update(path=old)))
            self.log.append(("place", row, old))

    def _entries(self, relation):
        return self.relations.setdefault(relation.name, [])

    # --- its reads ---

    def candidates(self, schema):
        name = self.store.name_of(schema)
        return [row for row in self.objects.values() if row["schema"] == name]

    def named(self, schema, name):
        return []  # it holds no schema objects: its schemas are its store's

    def designated(self, path):
        found = self.paths.get(json.dumps(path, sort_keys=True))
        return None if found is None else self.objects.get(found)

    def schema_of(self, target):
        return self.store.registered(target["schema"])

    def key(self, target):
        return target["id"]

    def value(self, holder, schema, name):
        values = holder["values"]
        return (True, values[name]) if name in values else (False, None)

    def entries(self, target, schema, name):
        adjacency = schema.adjacencies[name]
        return [entry for entry in self._entries(adjacency.relation) if entry["links"].get(adjacency.me) == target["id"]]

    def link(self, entry, name):
        return self.objects.get(entry["links"].get(name))

    def linking(self, target, relation):
        return [entry for entry in self._entries(relation) if target["id"] in entry["links"].values()]

    def part(self, entry, name):
        value = entry["values"].get(name)
        return None if value is None else json.dumps(value, sort_keys=True)

    def shown(self, entry, names):
        def one(name):
            target = self.link(entry, name)
            return f"<{target['schema']}>" if target is not None else repr(entry["values"].get(name))
        return "{" + ", ".join(f"{name}: {one(name)}" for name in names) + "}"

    def given(self, target, names):
        if not isinstance(target, dict):
            return target, {}  # a schema object, its store's
        if target["path"] is None:
            return Transactions.Refused(f"a counted {target['schema']!r} has no reference", target, self.life), {}
        reference = Plain.FromPlain(self.store)(self.store.registered("References.OfObject"), {"path": target["path"]})
        return reference, {name: target["values"][name] for name in names if name in target["values"]}

    # --- its writes, each logged to be undone ---

    def assign(self, target, schema, name, verb):
        values, had = target["values"], dict(target["values"])
        self.log.append(("assign", target, name, self.value(target, schema, name)))
        if "set" in verb:
            values[name] = verb["set"]["value"]
        else:
            values.pop(name, None)
        self.undo.append(lambda: (values.clear(), values.update(had)))

    def insert(self, target, schema, name, links, values):
        adjacency = schema.adjacencies[name]
        relation = adjacency.relation
        held = self._entries(relation)
        if Transactions.single(relation, adjacency.me) and relation.name not in self.missed:
            self.remove(target, schema, name, list(range(len(self.entries(target, schema, name)))))
        entry = {"links": {adjacency.me: target["id"], **{link: row["id"] for link, row in links.items()}},
                 "values": dict(values)}
        held.append(entry)
        self.inserted.append((relation, entry))
        self.log.append(("insert", relation, entry, target))
        self.undo.append(lambda: held.remove(entry))
        for link, row in links.items():  # an object it owns is placed: its path its owner's, one step on
            owned = [a for a in self.schema_of(row).adjacencies.values() if a.owner and a.relation is relation
                     and a.me == link]
            if owned and target["path"] is not None:
                step = {"entry": {"i": name, "o": owned[0].name, "properties": dict(values), "next": {"target": {}}}}
                self._place(row, {**target["path"], "next": _deeper(target["path"]["next"], step)})

    def remove(self, target, schema, name, positions):
        relation = schema.adjacencies[name].relation
        if not positions:
            self.missed.add(relation.name)
            return
        held = self._entries(relation)
        mine = self.entries(target, schema, name)
        for index in sorted(positions, reverse=True):
            entry = mine[index]
            at = held.index(entry)
            held.pop(at)
            self.undo.append(lambda at=at, entry=entry: held.insert(at, entry))
            self.log.append(("remove", relation, entry, target))

    def create(self, schema, values, within=None):
        return self._row(self.store.name_of(schema), values={name: verb["set"]["value"] for name, verb in values.items()})

    def upserted(self, index, made=None):
        if made is not None:
            self.counted[index] = made
        return self.counted.get(index)

    # --- its transaction ---

    @contextlib.contextmanager
    def savepoint(self, journal=None):
        mark, logged, changed = len(self.undo), len(self.log), []
        try:
            yield changed
        except BaseException:
            self._undo(mark)
            del self.log[logged:]
            raise
        if journal is not None:
            changed.append(self.changes(self.log[logged:], journal()))

    def changes(self, log, uid):
        """What a stretch of its log changed, net, as a `Changes.Transaction` of id `uid`, gathered by what every layer
        shares (`Changes.gather`): the rows it made, the entries it removed and inserted, the properties it changed,
        each object held before by its path as it was. It deletes nothing."""
        made, inserted, removed, assigned, paths = [], [], [], {}, {}
        for kind, *record in log:
            if kind == "create":
                made.append(record[0])
                for name in record[0]["values"]:  # each value it was made with: held by none before
                    assigned.setdefault((record[0]["id"], name), (record[0], name, (False, None)))
            elif kind == "place":
                paths.setdefault(record[0]["id"], record[1])
            elif kind == "insert":
                inserted.append(tuple(record[:2]))
            elif kind == "remove":
                if any(entry is record[1] for _, entry in inserted):
                    inserted = [(r, entry) for r, entry in inserted if entry is not record[1]]
                else:
                    removed.append(tuple(record[:2]))
            else:
                row, name, before = record
                assigned.setdefault((row["id"], name), (row, name, before))
        changed = [(row, name) for row, name, before in assigned.values() if self.value(row, None, name) != before]
        return Changes.gather(self, uid, made, [], removed, inserted, changed,
                              lambda row: paths.get(row["id"], row["path"]))

    def _undo(self, mark):
        while len(self.undo) > mark:
            self.undo.pop()()

    def uid(self):
        return self.txn_uid

    @contextlib.contextmanager
    def transaction(self, uid="t/1"):
        """The layer's transaction: `txn`, whose `exec` this layer's interpreter runs; at its end, the unique clauses
        of what it inserted are checked, and it commits, or, where it raises, every write is undone."""
        self.undo, self.inserted, self.missed, self.txn_uid, self.log = [], [], set(), uid, []
        self.life = SimpleNamespace(ended=False)
        txn = Transactions.Transaction(self, self.life)
        try:
            yield txn
            txn._raise()
            made = [record[1] for record in self.log if record[0] == "create"]
            written = {id(record[-1]): record[-1] for record in self.log if record[0] in ("insert", "remove")}
            written.update({id(record[1]): record[1] for record in self.log if record[0] == "assign"})
            Provenance.record(self, None, made, [row for row in written.values() if row not in made])
            by_relation = {}
            for relation, entry in self.inserted:
                if relation.uniques and entry in self._entries(relation):
                    by_relation.setdefault(relation.name, (relation, []))[1].append(entry)
            for relation, entries in by_relation.values():
                self.check_uniques(relation, entries)
        except BaseException:
            self._undo(0)
            raise
        finally:
            self.life.ended, self.undo = True, None


def _deeper(nested, step):
    """A path's `next`, its innermost `{"target": {}}` given a step more."""
    if "target" in nested:
        return step
    return {"entry": {**nested["entry"], "next": _deeper(nested["entry"]["next"], step)}}


def crm(txn):
    """crm.Person, owned by the singleton crm.TheBook under a key, who knows persons; crm.Note, counted."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Knows": S.OfRelation, "Person": S.OfObject, "Holds": S.OfRelation, "Book": S.OfObject,
                  "Note": S.OfObject})
        .spec([
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")]),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr, "age": S.OfInt}).adjacencies([
                lambda t: t.name("knows").relation([_.Knows]).me("knower"),
                lambda t: t.relation([_.Knows]).me("known"),
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True),
            ]),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")]).properties({
                "key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Note.name("crm.Note").ref(True).properties({"text": S.OfStr}),
        ])]))])


def people_of(t):
    """Ada, 36, who knows Bob, and Bob, of no age: crm.TheBook's contacts "ada" and "bob"."""
    return t.uid("people/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
        {"a": t.crm.Person, "b": t.crm.Person, "book": t.crm.Book}).spec([
            lambda _: _.book.contacts({"ada": lambda e: e.contact(_.a), "bob": lambda e: e.contact(_.b)}),
            lambda _: _.a.name("Ada").age(36).knows([_.b]),
            lambda _: _.b.name("Bob"),
        ])]))])


def pair():
    """A store in memory holding crm's schemas, Ada and Bob; and a layer of plain dictionaries over its schemas, given
    the same people by its own interpreter."""
    store = B.Store()
    store.transaction(crm)
    layer = Indexed(store)
    store.transaction(people_of)
    run(layer, Frame(), people_of, True)
    return store, layer


def expression(store, spec):
    """A transaction's expression, built over the store's schemas."""
    return store.expressed(spec)


def run(layer, frame, spec, allow_write):
    """The items of `txn.exec`'s stream in a layer of plain dictionaries, within a transaction of their own."""
    with layer.transaction() as txn:
        return list(txn.exec(frame, expression(layer.store, spec), allow_write))


def frames(items):
    """The frames of a stream's items, its completion markers left out."""
    return [item for item in items if isinstance(item, Frame)]


def paths(store, given):
    """Each frame's bindings, by symbol, as their paths, and what it assigns."""
    return [({symbol: json.dumps(Plain.ToPlain(store)(target.as_schema(), target)["path"], sort_keys=True)
              for symbol, target in sorted(frame.items())}, frame.assigns) for frame in given]


def people(store, layer):
    """Each person, the store's and the layer's: name, age and whom they know, by name."""
    with store.transaction():
        held = sorted((str(p.name), store.member(p, "age"), sorted(str(e.known.name) for e in p.knows))
                      for p in store.extent("crm.Person"))
    Person = store.registered("crm.Person")
    rows = layer.candidates(Person)
    kept = sorted((row["values"]["name"], row["values"].get("age"), sorted(
        layer.link(entry, "known")["values"]["name"] for entry in layer.entries(row, Person, "knows"))) for row in rows)
    return held, kept


def ada_then(then):
    """A transaction: a map binding Ada to x, then the term `then` gives."""
    def build(txn):
        Person = txn.crm.Person
        return txn.uid("ada/1").mutations([lambda mu: mu.map(lambda m: m.symbols([
            lambda p: p.symbols({"x": Person}).spec([lambda _: _.x.name("Ada")])]).then([then(Person)]))])
    return build


def aged(age):
    """An update writing x's age."""
    return lambda Person: lambda w: w.update(lambda u: u.apply([
        lambda p: p.symbols({"x": Person}).spec([lambda _: _.x.age(age)])]))

<a id="LAY-01"></a>
## LAY-01 · The base class and a layer's: `Transactions.Interpreter` holds the walk and asks a layer for its reads, writes and savepoints, each a method a layer's class must give, so that a class that gives none cannot be made; a store in memory's interpreter is `Transactions.InMemory`, one for the store, coupled with it, which its transactions' `exec` runs

In [ ]:
with raises(TypeError, match="abstract"):
    Transactions.Interpreter(B.Store())  # what a layer gives, it must give
store = B.Store()
assert isinstance(store._interpreter(), Transactions.InMemory) and store._interpreter() is store._interpreter()
assert store._interpreter().store is store and not store._interpreter().cache  # no layer below: it holds all it holds
with store.transaction() as txn:
    assert txn._interpreter is store._interpreter()
assert B.Store(store)._interpreter().cache  # a cache layer's
hooks = {"candidates", "named", "designated", "schema_of", "key", "value", "entries", "link", "linking", "part", "shown",
         "given", "assign", "insert", "remove", "create", "upserted", "savepoint", "uid"}
assert Transactions.Interpreter.__abstractmethods__ == hooks

<a id="LAY-02"></a>
## LAY-02 · A layer not built on proxies interprets as a store in memory does: its data plain dictionaries, its reads and writes its own, the base class drives an upsert that makes owned and counted objects, run again finding them; a map through entries and properties; an update; a reduce; a partition's branches joined; its frames, by their references, and its data, the same as a store in memory's

In [ ]:
store, layer = pair()
assert people(store, layer) == ([("Ada", 36, ["Bob"]), ("Bob", None, [])],) * 2
assert not layer.cache and all(layer.knows(row) for row in layer.objects.values())  # it holds all it holds
assert sorted(layer.paths) == sorted(json.dumps(p, sort_keys=True) for p in [
    {"singleton": "crm.TheBook", "next": {"target": {}}},
    *({"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": key},
                                                       "next": {"target": {}}}}} for key in ("ada", "bob"))])
again = frames(run(layer, Frame(), people_of, True))  # run again: it finds what it made, writing nothing
assert paths(store, again) == paths(store, store.transaction(people_of)) and len(layer.candidates(
    store.registered("crm.Person"))) == 2
note = lambda t: t.uid("note/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"n": t.crm.Note}).spec([lambda _: _.n.text("x")])]))])
(first,) = frames(run(layer, Frame(), note, True))
with raises(LookupError, match="has no reference"):
    first["n"]  # a counted object: refused, as a store in memory refuses it
(n,) = layer.candidates(store.registered("crm.Note"))
with layer.transaction("t/1") as txn:  # the same transaction's uid: the same counted object, found by what made it
    list(txn.exec(Frame(), expression(store, note), True))
    list(txn.exec(Frame(), expression(store, note), True))
with layer.transaction("t/2") as txn:  # another's: another
    list(txn.exec(Frame(), expression(store, note), True))
assert [row["values"] for row in layer.candidates(store.registered("crm.Note"))] == [{"text": "x"}] * 2
knowing = lambda t: t.uid("knowing/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": t.crm.Person, "y": t.crm.Person}).spec([lambda _: _.x.knows([_.y]), lambda _: _.y.name("Bob")])]))])
for spec in (knowing, ada_then(aged(40)), ada_then(aged(None))):
    assert paths(store, frames(run(layer, Frame(), spec, True))) == paths(store, store.transaction(spec))
assert people(store, layer)[0] == people(store, layer)[1] == [("Ada", None, ["Bob"]), ("Bob", None, [])]
pairs = lambda t: t.uid("pairs/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": t.crm.Person, "y": t.crm.Person})])).then([lambda w: w.reduce(lambda r: r.symbols([
        lambda c: c.symbols(["y"])]))])])
split = lambda t: t.uid("split/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": t.crm.Person})])).then([lambda w: w.partition(lambda q: q.branches([
        lambda e: e.branch(lambda b: b.then([lambda w2: w2.map(lambda m2: m2.symbols([lambda p: p.symbols(
            {"x": t.crm.Person}).spec([lambda _: _.x.name("Ada")])]))])),
        lambda e: e.branch(lambda b: b.then([lambda w2: w2.update(lambda u: u.apply([lambda p: p.symbols(
            {"x": t.crm.Person}).spec([lambda _: _.x.age(7)])]))])),
    ]))])])
for spec in (pairs, split):
    mine, theirs = paths(store, frames(run(layer, Frame(), spec, True))), paths(store, store.transaction(spec))
    assert sorted(map(repr, mine)) == sorted(map(repr, theirs)) and len(mine) == (2 if spec is pairs else 3)
assert people(store, layer)[0] == people(store, layer)[1] == [("Ada", 7, ["Bob"]), ("Bob", 7, [])]

<a id="LAY-03"></a>
## LAY-03 · A layer not built on proxies streams as a store in memory does: `txn.exec` over input frames binding symbols by their references, each frame carrying its input frame's id, then its completion marker; read-only, a frame halting where a term would write, given with its remainder, which the same transaction completes writing, the loopback; a partition's join kept through the loopback; an error annotated on its frame, the scope then raising it and undoing every write

In [ ]:
store, layer = pair()
everyone = lambda t: t.uid("everyone/1").mutations([lambda mu: mu.map(lambda m: m.symbols([
    lambda p: p.symbols({"x": t.crm.Person})]))])
older = lambda t: t.uid("older/1").mutations([lambda mu: mu.update(lambda u: u.apply([
    lambda p: p.symbols({"x": t.crm.Person}).spec([lambda _: _.x.age(50)])]))])
given = frames(run(layer, Frame(), everyone, True))
items = run(layer, [{"x": frame["x"]} for frame in given], older, True)  # input frames by their references
assert [item.scope if isinstance(item, Frame) else item for item in items] == ["0", {"end": "0"}, "1", {"end": "1"}]
assert [frame.assigns for frame in frames(items)] == [{"x": {"age": 50}}] * 2
assert people(store, layer)[1] == [("Ada", 50, ["Bob"]), ("Bob", 50, [])]
with layer.transaction() as txn:
    (halted,) = frames(txn.exec(Frame(), expression(store, ada_then(aged(60))), False))  # read-only: it would write
    assert halted.remainder.subterm == "$.mutations[0].map.then[0].update" and people(store, layer)[1][0][1] == 50
    (looped,) = frames(txn.exec(halted, halted.remainder, True))  # the loopback: the same transaction, writing
assert looped.assigns == {"x": {"name": "Ada", "age": 60}} and people(store, layer)[1][0] == ("Ada", 60, ["Bob"])
split = lambda t: t.uid("split/2").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": t.crm.Person}).spec([lambda _: _.x.name("Bob")])])).then([lambda w: w.partition(lambda q: q.branches([
        lambda e: e.branch(lambda b: b.then([lambda w2: w2.update(lambda u: u.apply([lambda p: p.symbols(
            {"x": t.crm.Person}).spec([lambda _: _.x.age(3)])]))]))]).then([lambda w3: w3.map(lambda m3: m3.symbols([
                lambda p: p.symbols({"bk": t.crm.Book})]))]))])])
items = run(layer, Frame(), split, False)  # read-only, it halts within the branch: the loopback writes, joined
(joined,) = frames(items)
assert joined.remainder is None and sorted(joined) == ["bk", "x"] and people(store, layer)[1][1] == ("Bob", 3, [])
nobody = {"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": "zed"},
                                                         "next": {"target": {}}}}}
zed = Plain.FromPlain(store)(store.registered("References.OfObject"), {"path": nobody})
with raises(LookupError, match="no object of the store has its reference's path"):
    with layer.transaction() as txn:
        items = list(txn.exec([{"x": joined["x"]}, {"x": zed}], expression(store, older), True))
        assert [bool(frame.errors) for frame in frames(items)] == [False, True]  # annotated, nothing raised by exec
        assert frames(items)[1].errors[0].subterm == "$.mutations[0].update"
assert people(store, layer)[1][1] == ("Bob", 3, [])  # the scope raised: every write undone

<a id="LAY-04"></a>
## LAY-04 · Unique clauses, the base class's, checked at a layer's commit over what it inserted: a write that leaves two entries agreeing on a clause's key refused, whole, with the message a store in memory gives; removing the entry held first, accepted

In [ ]:
store, layer = pair()
twice = lambda t: t.uid("twice/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": t.crm.Person, "book": t.crm.Book}).spec([lambda _: _.x.name("Bob")])])).then([lambda w: w.update(
        lambda u: u.apply([lambda p: p.symbols({"x": t.crm.Person, "book": t.crm.Book}).spec([
            lambda _: _.book.contacts({"ada": lambda e: e.contact(_.x)})])])).then([lambda w2: w2.reduce(
                lambda r: r.symbols([lambda c: c.symbols(["x"])]))])])])  # Bob under Ada's key too; the book kept
def refusal(call):
    """The message of the `ValueError` a call raises."""
    try:
        call()
    except ValueError as error:
        return str(error)
    raise AssertionError("expected a refusal")


theirs, mine = refusal(lambda: store.transaction(twice)), refusal(lambda: run(layer, Frame(), twice, True))
assert mine == theirs and "breaks unique(" in mine
assert len(layer.relations["crm.Holds"]) == 2  # refused, whole
moved = lambda t: t.uid("moved/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": t.crm.Person, "book": t.crm.Book}).spec([lambda _: _.x.name("Bob")])])).then([lambda w: w.update(
        lambda u: u.apply([lambda p: p.symbols({"x": t.crm.Person, "book": t.crm.Book}).spec([
            lambda _: _.book.contacts(lambda a: a.remove(lambda e: e.contact(_.x))),
            lambda _: _.book.contacts({"robert": lambda e: e.contact(_.x)})])])).then([lambda w2: w2.reduce(
                lambda r: r.symbols([lambda c: c.symbols(["x"])]))])])])
assert paths(store, frames(run(layer, Frame(), moved, True))) == paths(store, store.transaction(moved))
assert sorted(entry["values"]["key"] for entry in layer.relations["crm.Holds"]) == ["ada", "robert"]

<a id="LAY-05"></a>
## LAY-05 · What every layer shares, public: a frame's journal gathered from what the layer changed by `Changes.gather`, the same as a store in memory's; its transactions recorded by `Provenance.record`, found again by their uid, an object's `ModifiedBy` replaced; `Transactions.Refused`, `Interpreter.check_uniques` and `Transactions.single`, which its `given`, its commit and its `insert` use; the schema a path implies, `References.implied`; and a callback, given the store's schemas by name, built by `store.expressed` as `txn.exec` builds one, so that `Changes.apply` writes a journal through any layer

In [ ]:
store, layer = pair()
schema = store.registered("Changes.Journal")
cy = lambda t: t.uid("cy/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"c": t.crm.Person, "a": t.crm.Person, "book": t.crm.Book}).spec([
        lambda _: _.book.contacts({"cy": lambda e: e.contact(_.c)}), lambda _: _.c.name("Cy").knows([_.a]),
        lambda _: _.a.name("Ada")])]))])
moved = lambda t: t.uid("moved/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": t.crm.Person, "book": t.crm.Book}).spec([lambda _: _.x.name("Bob")])])).then([lambda w: w.update(
        lambda u: u.apply([lambda p: p.symbols({"x": t.crm.Person, "book": t.crm.Book}).spec([
            lambda _: _.book.contacts(lambda a: a.remove(lambda e: e.contact(_.x))),
            lambda _: _.book.contacts({"robert": lambda e: e.contact(_.x)}), lambda _: _.x.age(9)])]))])])
for spec, uid in ((ada_then(aged(40)), "ada/1"), (ada_then(aged(None)), "ada/1"), (cy, "cy/1"), (moved, "moved/1")):
    with store.transaction() as txn:  # a callback, built by the store as `txn.exec` is given it
        theirs = [Plain.ToPlain(store)(schema, frame.journal) for frame in frames(txn.exec(Frame(), spec, True))]
    with layer.transaction(uid) as txn:
        mine = [Plain.ToPlain(store)(schema, frame.journal) for frame in frames(txn.exec(Frame(), spec, True))]
    assert mine == theirs and theirs[0][0]["id"] == f"{uid}#0"
assert people(store, layer)[0] == people(store, layer)[1]
with layer.transaction() as txn:  # nothing changed: no journal
    assert [frame.journal for frame in frames(txn.exec(Frame(), ada_then(aged(None)), True))] == [None]

replica, got = B.Store(), []  # a journal, written through a layer by `Changes.apply` within a callback
replica.transaction(crm)
Changes.observe(replica, got.append)
replica.transaction(people_of)
fresh = Indexed(replica)
run(fresh, Frame(), lambda t: Changes.apply(t, got[0]), True)
assert people(replica, fresh) == ([("Ada", 36, ["Bob"]), ("Bob", None, [])],) * 2

kept = B.Store()  # its transactions recorded, as every layer's are
kept.transaction(crm)
kept.transaction(Provenance.install)
kept.transaction(lambda t: Provenance.extend(t, [t.crm.Person]))
recorded = Indexed(kept)
for uid, spec in (("p/1", people_of), ("a/1", ada_then(aged(1))), ("a/1", ada_then(aged(2))), ("a/2", ada_then(aged(3)))):
    with recorded.transaction(uid) as txn:
        list(txn.exec(Frame(), spec, True))
by_uid = lambda schema_name, adjacency, key: [entry["values"][key] for entry in recorded.entries(  # noqa: E731
    recorded.candidates(kept.registered(schema_name))[0], kept.registered(schema_name), adjacency)]
assert by_uid("Provenance.TransactionDirectory", "transactions", "uid") == ["p/1", "a/1", "a/2"]
assert by_uid("Provenance.SessionDirectory", "sessions", "id") == ["unknown"]
assert by_uid("Provenance.IdentityDirectory", "identities", "name") == ["unknown"]
records = {entry["values"]["uid"]: recorded.link(entry, "transaction") for entry in recorded.entries(
    recorded.candidates(kept.registered("Provenance.TransactionDirectory"))[0],
    kept.registered("Provenance.TransactionDirectory"), "transactions")}
OfTransaction = kept.registered("Provenance.OfTransaction")
named = lambda record, adjacency: sorted(recorded.link(e, "object")["values"]["name"]  # noqa: E731
                                         for e in recorded.entries(record, OfTransaction, adjacency))
assert named(records["p/1"], "created") == ["Ada", "Bob"] and named(records["a/2"], "modified") == ["Ada"]
assert named(records["a/1"], "modified") == []  # replaced: the tail, not a history
assert [len(recorded.entries(record, OfTransaction, "session")) for record in records.values()] == [1, 1, 1]

Person = store.registered("crm.Person")  # what its given, its commit and its insert use
assert References.implied(store, {"singleton": "crm.TheBook", "next": {"target": {}}}) is store.registered("crm.Book")
ada = {"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": "ada"},
                                                      "next": {"target": {}}}}}
assert References.implied(store, ada) is Person
with raises(LookupError, match="no schema declares a singleton"):
    References.implied(store, {"singleton": "crm.Nobody", "next": {"target": {}}})
Holds = store.registered("crm.Holds")
assert Transactions.single(Holds, "contact") and not Transactions.single(Holds, "book")
refused = Transactions.Refused("no reference")
with raises(LookupError, match="no reference"):
    refused.name
assert {"Refused", "single"} <= set(Transactions.__all__) and "check_uniques" in vars(Transactions.Interpreter)
assert "gather" in Changes.__all__ and "record" in Provenance.__all__ and "implied" in References.__all__
